# Лабораторная работа 4

Tensorflow 2.x

1) Подготовка данных

2) Использование Keras Model API

3) Использование Keras Sequential + Functional API

https://www.tensorflow.org/tutorials

Для выполнения лабораторной работы необходимо установить tensorflow версии 2.0 или выше .

Рекомендуется использовать возможности Colab'а по обучению моделей на GPU.



In [1]:
import os
import tensorflow as tf
import numpy as np
import math
import timeit
import matplotlib.pyplot as plt

%matplotlib inline

I0000 00:00:1777324325.207677    2224 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1777324345.980332    2224 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1777324358.942858    2224 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
device = '/GPU:0' if tf.config.list_physical_devices('GPU') else '/CPU:0'
print(device)

/CPU:0


E0000 00:00:1777324363.380609    2224 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


# Подготовка данных
Загрузите набор данных из предыдущей лабораторной работы. 

In [3]:
def load_cifar10(num_training=49000, num_validation=1000, num_test=10000):
    """
    Fetch the CIFAR-10 dataset from the web and perform preprocessing to prepare
    it for the two-layer neural net classifier. These are the same steps as
    we used for the SVM, but condensed to a single function.
    """
    # Load the raw CIFAR-10 dataset and use appropriate data types and shapes
    cifar10 = tf.keras.datasets.cifar10.load_data()
    (X_train, y_train), (X_test, y_test) = cifar10
    X_train = np.asarray(X_train, dtype=np.float32)
    y_train = np.asarray(y_train, dtype=np.int32).flatten()
    X_test = np.asarray(X_test, dtype=np.float32)
    y_test = np.asarray(y_test, dtype=np.int32).flatten()

    # Subsample the data
    mask = range(num_training, num_training + num_validation)
    X_val = X_train[mask]
    y_val = y_train[mask]
    mask = range(num_training)
    X_train = X_train[mask]
    y_train = y_train[mask]
    mask = range(num_test)
    X_test = X_test[mask]
    y_test = y_test[mask]

    # Normalize the data: subtract the mean pixel and divide by std
    mean_pixel = X_train.mean(axis=(0, 1, 2), keepdims=True)
    std_pixel = X_train.std(axis=(0, 1, 2), keepdims=True)
    X_train = (X_train - mean_pixel) / std_pixel
    X_val = (X_val - mean_pixel) / std_pixel
    X_test = (X_test - mean_pixel) / std_pixel

    return X_train, y_train, X_val, y_val, X_test, y_test

# If there are errors with SSL downloading involving self-signed certificates,
# it may be that your Python version was recently installed on the current machine.
# See: https://github.com/tensorflow/tensorflow/issues/10779
# To fix, run the command: /Applications/Python\ 3.7/Install\ Certificates.command
#   ...replacing paths as necessary.

# Invoke the above function to get our data.
NHW = (0, 1, 2)
X_train, y_train, X_val, y_val, X_test, y_test = load_cifar10()
print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape, y_train.dtype)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/datasets/cifar.py:18: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  d = cPickle.load(f, encoding="bytes")


Train data shape:  (49000, 32, 32, 3)
Train labels shape:  (49000,) int32
Validation data shape:  (1000, 32, 32, 3)
Validation labels shape:  (1000,)
Test data shape:  (10000, 32, 32, 3)
Test labels shape:  (10000,)


In [4]:
class Dataset(object):
    def __init__(self, X, y, batch_size, shuffle=False):
        """
        Construct a Dataset object to iterate over data X and labels y
        
        Inputs:
        - X: Numpy array of data, of any shape
        - y: Numpy array of labels, of any shape but with y.shape[0] == X.shape[0]
        - batch_size: Integer giving number of elements per minibatch
        - shuffle: (optional) Boolean, whether to shuffle the data on each epoch
        """
        assert X.shape[0] == y.shape[0], 'Got different numbers of data and labels'
        self.X, self.y = X, y
        self.batch_size, self.shuffle = batch_size, shuffle

    def __iter__(self):
        N, B = self.X.shape[0], self.batch_size
        idxs = np.arange(N)
        if self.shuffle:
            np.random.shuffle(idxs)
        return iter((self.X[i:i+B], self.y[i:i+B]) for i in range(0, N, B))


train_dset = Dataset(X_train, y_train, batch_size=64, shuffle=True)
val_dset = Dataset(X_val, y_val, batch_size=64, shuffle=False)
test_dset = Dataset(X_test, y_test, batch_size=64)

In [5]:
# We can iterate through a dataset like this:
for t, (x, y) in enumerate(train_dset):
    print(t, x.shape, y.shape)
    if t > 5: break

0 (64, 32, 32, 3) (64,)
1 (64, 32, 32, 3) (64,)
2 (64, 32, 32, 3) (64,)
3 (64, 32, 32, 3) (64,)
4 (64, 32, 32, 3) (64,)
5 (64, 32, 32, 3) (64,)
6 (64, 32, 32, 3) (64,)


#  Keras Model Subclassing API


Для реализации собственной модели с помощью Keras Model Subclassing API необходимо выполнить следующие шаги:

1) Определить новый класс, который является наследником tf.keras.Model.

2) В методе __init__() определить все необходимые слои из модуля tf.keras.layer

3) Реализовать прямой проход в методе call() на основе слоев, объявленных в __init__()

Ниже приведен пример использования keras API для определения двухслойной полносвязной сети. 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras

In [6]:
class TwoLayerFC(tf.keras.Model):
    def __init__(self, hidden_size, num_classes):
        super(TwoLayerFC, self).__init__()        
        initializer = tf.initializers.VarianceScaling(scale=2.0)
        self.fc1 = tf.keras.layers.Dense(hidden_size, activation='relu',
                                   kernel_initializer=initializer)
        self.fc2 = tf.keras.layers.Dense(num_classes, activation='softmax',
                                   kernel_initializer=initializer)
        self.flatten = tf.keras.layers.Flatten()
    
    def call(self, x, training=False):
        x = self.flatten(x)
        x = self.fc1(x)
        x = self.fc2(x)
        return x


def test_TwoLayerFC():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    x = tf.zeros((64, input_size))
    model = TwoLayerFC(hidden_size, num_classes)
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_TwoLayerFC()

(64, 10)


Реализуйте трехслойную CNN для вашей задачи классификации. 

Архитектура сети:
    
1. Сверточный слой (5 x 5 kernels, zero-padding = 'same')
2. Функция активации ReLU 
3. Сверточный слой (3 x 3 kernels, zero-padding = 'same')
4. Функция активации ReLU 
5. Полносвязный слой 
6. Функция активации Softmax 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Conv2D

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dense

In [7]:
class ThreeLayerConvNet(tf.keras.Model):
    def __init__(self, channel_1, channel_2, num_classes):
        super(ThreeLayerConvNet, self).__init__()
        ########################################################################
        # TODO: Implement the __init__ method for a three-layer ConvNet. You   #
        # should instantiate layer objects to be used in the forward pass.     #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(
            filters=channel_1,
            kernel_size=(5, 5),
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        )

        self.conv2 = tf.keras.layers.Conv2D(
            filters=channel_2,
            kernel_size=(3, 3),
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        )

        self.flatten = tf.keras.layers.Flatten()

        self.fc = tf.keras.layers.Dense(
            num_classes,
            activation='softmax',
            kernel_initializer=initializer
        )

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################
        
    def call(self, x, training=False):
        scores = None
        ########################################################################
        # TODO: Implement the forward pass for a three-layer ConvNet. You      #
        # should use the layer objects defined in the __init__ method.         #
        ########################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        if len(x.shape) == 4 and x.shape[1] == 3:
            x = tf.transpose(x, [0, 2, 3, 1])

        x = self.conv1(x)
        x = self.conv2(x)
        x = self.flatten(x)
        scores = self.fc(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ########################################################################
        #                           END OF YOUR CODE                           #
        ########################################################################        
        return scores

In [8]:
def test_ThreeLayerConvNet():    
    channel_1, channel_2, num_classes = 12, 8, 10
    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)
    with tf.device(device):
        x = tf.zeros((64, 3, 32, 32))
        scores = model(x)
        print(scores.shape)

test_ThreeLayerConvNet()

(64, 10)


Пример реализации процесса обучения:

In [9]:
def train_part34(model_init_fn, optimizer_init_fn, num_epochs=1, is_training=False):
    """
    Simple training loop for use with models defined using tf.keras. It trains
    a model for one epoch on the CIFAR-10 training set and periodically checks
    accuracy on the CIFAR-10 validation set.
    
    Inputs:
    - model_init_fn: A function that takes no parameters; when called it
      constructs the model we want to train: model = model_init_fn()
    - optimizer_init_fn: A function which takes no parameters; when called it
      constructs the Optimizer object we will use to optimize the model:
      optimizer = optimizer_init_fn()
    - num_epochs: The number of epochs to train for
    
    Returns: Nothing, but prints progress during trainingn
    """    
    with tf.device(device):

        
        loss_fn = tf.keras.losses.SparseCategoricalCrossentropy()
        
        model = model_init_fn()
        optimizer = optimizer_init_fn()
        
        train_loss = tf.keras.metrics.Mean(name='train_loss')
        train_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='train_accuracy')
    
        val_loss = tf.keras.metrics.Mean(name='val_loss')
        val_accuracy = tf.keras.metrics.SparseCategoricalAccuracy(name='val_accuracy')
        
        t = 0
        for epoch in range(num_epochs):
            
            # Reset the metrics - https://www.tensorflow.org/alpha/guide/migration_guide#new-style_metrics
            train_loss.reset_state()
            train_accuracy.reset_state()
            
            for x_np, y_np in train_dset:
                with tf.GradientTape() as tape:
                    
                    # Use the model function to build the forward pass.
                    scores = model(x_np, training=is_training)
                    loss = loss_fn(y_np, scores)
      
                    gradients = tape.gradient(loss, model.trainable_variables)
                    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
                    
                    # Update the metrics
                    train_loss.update_state(loss)
                    train_accuracy.update_state(y_np, scores)
                    
                    if t % print_every == 0:
                        val_loss.reset_state()
                        val_accuracy.reset_state()
                        for test_x, test_y in val_dset:
                            # During validation at end of epoch, training set to False
                            prediction = model(test_x, training=False)
                            t_loss = loss_fn(test_y, prediction)

                            val_loss.update_state(t_loss)
                            val_accuracy.update_state(test_y, prediction)
                        
                        template = 'Iteration {}, Epoch {}, Loss: {}, Accuracy: {}, Val Loss: {}, Val Accuracy: {}'
                        print (template.format(t, epoch+1,
                                             train_loss.result(),
                                             train_accuracy.result()*100,
                                             val_loss.result(),
                                             val_accuracy.result()*100))
                    t += 1

In [13]:
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2
print_every=100

def model_init_fn():
    return TwoLayerFC(hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.08097505569458, Accuracy: 9.375, Val Loss: 2.8557984828948975, Val Accuracy: 12.100000381469727
Iteration 100, Epoch 1, Loss: 2.217179298400879, Accuracy: 28.140470504760742, Val Loss: 1.8754315376281738, Val Accuracy: 36.599998474121094
Iteration 200, Epoch 1, Loss: 2.0615949630737305, Accuracy: 32.40049743652344, Val Loss: 1.8147016763687134, Val Accuracy: 38.900001525878906
Iteration 300, Epoch 1, Loss: 1.9920399188995361, Accuracy: 34.265987396240234, Val Loss: 1.8370163440704346, Val Accuracy: 38.5
Iteration 400, Epoch 1, Loss: 1.9269371032714844, Accuracy: 36.01153564453125, Val Loss: 1.7175508737564087, Val Accuracy: 41.80000305175781
Iteration 500, Epoch 1, Loss: 1.883057713508606, Accuracy: 37.03842544555664, Val Loss: 1.6478301286697388, Val Accuracy: 42.69999694824219
Iteration 600, Epoch 1, Loss: 1.853400707244873, Accuracy: 37.94197082519531, Val Loss: 1.692535400390625, Val Accuracy: 41.20000076293945
Iteration 700, Epoch 1, Loss: 1.828039526

Обучите трехслойную CNN. В tf.keras.optimizers.SGD укажите Nesterov momentum = 0.9 . 

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/optimizers/SGD

Значение accuracy на валидационной выборке после 1 эпохи обучения должно быть > 50% .

In [14]:
learning_rate = 3e-3
channel_1, channel_2, num_classes = 32, 16, 10
print_every=100

def model_init_fn():
    model = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    model = ThreeLayerConvNet(channel_1, channel_2, num_classes)

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return model

def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 2.5354604721069336, Accuracy: 10.9375, Val Loss: 2.7651329040527344, Val Accuracy: 9.700000762939453
Iteration 100, Epoch 1, Loss: 1.829620361328125, Accuracy: 34.97834014892578, Val Loss: 1.615973949432373, Val Accuracy: 43.79999923706055
Iteration 200, Epoch 1, Loss: 1.6793197393417358, Accuracy: 40.71051025390625, Val Loss: 1.454566240310669, Val Accuracy: 49.20000076293945
Iteration 300, Epoch 1, Loss: 1.5966742038726807, Accuracy: 43.75, Val Loss: 1.3952531814575195, Val Accuracy: 50.900001525878906
Iteration 400, Epoch 1, Loss: 1.528873085975647, Accuracy: 45.96711349487305, Val Loss: 1.3030495643615723, Val Accuracy: 54.000003814697266
Iteration 500, Epoch 1, Loss: 1.4775038957595825, Accuracy: 47.844932556152344, Val Loss: 1.2563674449920654, Val Accuracy: 55.599998474121094
Iteration 600, Epoch 1, Loss: 1.4418129920959473, Accuracy: 49.1264533996582, Val Loss: 1.2325164079666138, Val Accuracy: 57.5
Iteration 700, Epoch 1, Loss: 1.4115569591522217, A

# Использование Keras Sequential API для реализации последовательных моделей.

Пример для полносвязной сети:

In [15]:
learning_rate = 1e-2
print_every=100

def model_init_fn():
    input_shape = (32, 32, 3)
    hidden_layer_size, num_classes = 4000, 10
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    layers = [
        tf.keras.layers.Flatten(input_shape=input_shape),
        tf.keras.layers.Dense(hidden_layer_size, activation='relu',
                              kernel_initializer=initializer),
        tf.keras.layers.Dense(num_classes, activation='softmax', 
                              kernel_initializer=initializer),
    ]
    model = tf.keras.Sequential(layers)
    return model

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate) 

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Iteration 0, Epoch 1, Loss: 3.1869726181030273, Accuracy: 7.8125, Val Loss: 3.0925793647766113, Val Accuracy: 12.800000190734863
Iteration 100, Epoch 1, Loss: 2.2079052925109863, Accuracy: 29.42450714111328, Val Loss: 1.8767021894454956, Val Accuracy: 38.70000076293945
Iteration 200, Epoch 1, Loss: 2.06219744682312, Accuracy: 32.89801025390625, Val Loss: 1.8389708995819092, Val Accuracy: 39.89999771118164
Iteration 300, Epoch 1, Loss: 1.9953820705413818, Accuracy: 34.48920440673828, Val Loss: 1.8412723541259766, Val Accuracy: 37.29999923706055
Iteration 400, Epoch 1, Loss: 1.9311145544052124, Accuracy: 36.120635986328125, Val Loss: 1.7108092308044434, Val Accuracy: 42.69999694824219
Iteration 500, Epoch 1, Loss: 1.8870372772216797, Accuracy: 37.11327362060547, Val Loss: 1.636325478553772, Val Accuracy: 45.0
Iteration 600, Epoch 1, Loss: 1.8591036796569824, Accuracy: 37.90037155151367, Val Loss: 1.6712743043899536, Val Accuracy: 43.20000076293945
Iteration 700, Epoch 1, Loss: 1.83248364

Альтернативный менее гибкий способ обучения:

In [16]:
model = model_init_fn()
model.compile(optimizer=tf.keras.optimizers.SGD(learning_rate=learning_rate),
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

W0000 00:00:1777324628.025738    2224 cpu_allocator_impl.cc:82] Allocation of 602112000 exceeds 10% of free system memory.


766/766 ━━━━━━━━━━━━━━━━━━━━ 24s 30ms/step - loss: 1.8172 - sparse_categorical_accuracy: 0.3886 - val_loss: 1.7233 - val_sparse_categorical_accuracy: 0.4040
313/313 ━━━━━━━━━━━━━━━━━━━━ 3s 8ms/step - loss: 1.6822 - sparse_categorical_accuracy: 0.4190


[1.6821845769882202, 0.4189999997615814]

Перепишите реализацию трехслойной CNN с помощью tf.keras.Sequential API . Обучите модель двумя способами.

In [17]:
def model_init_fn():
    model = None
    ############################################################################
    # TODO: Construct a three-layer ConvNet using tf.keras.Sequential.         #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    initializer = tf.initializers.VarianceScaling(scale=2.0)

    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(
            32,
            kernel_size=(5, 5),
            padding='same',
            activation='relu',
            kernel_initializer=initializer,
            input_shape=(32, 32, 3)
        ),
        tf.keras.layers.Conv2D(
            16,
            kernel_size=(3, 3),
            padding='same',
            activation='relu',
            kernel_initializer=initializer
        ),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(
            10,
            activation='softmax',
            kernel_initializer=initializer
        )
    ])

    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                            END OF YOUR CODE                              #
    ############################################################################
    return model

learning_rate = 5e-4
print_every=100
def optimizer_init_fn():
    optimizer = None
    ############################################################################
    # TODO: Complete the implementation of model_fn.                           #
    ############################################################################
    # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

    optimizer = tf.keras.optimizers.SGD(
        learning_rate=learning_rate,
        momentum=0.9,
        nesterov=True
    )
  
    # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
    ############################################################################
    #                           END OF YOUR CODE                               #
    ############################################################################
    return optimizer

train_part34(model_init_fn, optimizer_init_fn)

/usr/local/python/3.12.1/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Iteration 0, Epoch 1, Loss: 2.8080620765686035, Accuracy: 4.6875, Val Loss: 2.7931442260742188, Val Accuracy: 11.200000762939453
Iteration 100, Epoch 1, Loss: 1.9886813163757324, Accuracy: 28.790224075317383, Val Loss: 1.7675552368164062, Val Accuracy: 37.599998474121094
Iteration 200, Epoch 1, Loss: 1.8601551055908203, Accuracy: 34.211753845214844, Val Loss: 1.635815143585205, Val Accuracy: 43.5
Iteration 300, Epoch 1, Loss: 1.7909340858459473, Accuracy: 36.778446197509766, Val Loss: 1.5981011390686035, Val Accuracy: 44.400001525878906
Iteration 400, Epoch 1, Loss: 1.731981873512268, Accuracy: 38.89495086669922, Val Loss: 1.5411721467971802, Val Accuracy: 46.0
Iteration 500, Epoch 1, Loss: 1.690909504890442, Accuracy: 40.350547790527344, Val Loss: 1.5044962167739868, Val Accuracy: 47.79999923706055
Iteration 600, Epoch 1, Loss: 1.6646533012390137, Accuracy: 41.477745056152344, Val Loss: 1.4678902626037598, Val Accuracy: 47.400001525878906
Iteration 700, Epoch 1, Loss: 1.64024400711059

In [18]:
model = model_init_fn()
model.compile(optimizer='sgd',
              loss='sparse_categorical_crossentropy',
              metrics=[tf.keras.metrics.sparse_categorical_accuracy])
model.fit(X_train, y_train, batch_size=64, epochs=1, validation_data=(X_val, y_val))
model.evaluate(X_test, y_test)

W0000 00:00:1777324719.717126    2224 cpu_allocator_impl.cc:82] Allocation of 602112000 exceeds 10% of free system memory.


766/766 ━━━━━━━━━━━━━━━━━━━━ 37s 48ms/step - loss: 1.6364 - sparse_categorical_accuracy: 0.4241 - val_loss: 1.4099 - val_sparse_categorical_accuracy: 0.4900
313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 1.4031 - sparse_categorical_accuracy: 0.4954


[1.4031471014022827, 0.49540001153945923]

# Использование Keras Functional API

Для реализации более сложных архитектур сети с несколькими входами/выходами, повторным использованием слоев, "остаточными" связями (residual connections) необходимо явно указать входные и выходные тензоры. 

Ниже представлен пример для полносвязной сети. 

In [19]:
def two_layer_fc_functional(input_shape, hidden_size, num_classes):  
    initializer = tf.initializers.VarianceScaling(scale=2.0)
    inputs = tf.keras.Input(shape=input_shape)
    flattened_inputs = tf.keras.layers.Flatten()(inputs)
    fc1_output = tf.keras.layers.Dense(hidden_size, activation='relu',
                                 kernel_initializer=initializer)(flattened_inputs)
    scores = tf.keras.layers.Dense(num_classes, activation='softmax',
                             kernel_initializer=initializer)(fc1_output)

    # Instantiate the model given inputs and outputs.
    model = tf.keras.Model(inputs=inputs, outputs=scores)
    return model

def test_two_layer_fc_functional():
    """ A small unit test to exercise the TwoLayerFC model above. """
    input_size, hidden_size, num_classes = 50, 42, 10
    input_shape = (50,)
    
    x = tf.zeros((64, input_size))
    model = two_layer_fc_functional(input_shape, hidden_size, num_classes)
    
    with tf.device(device):
        scores = model(x)
        print(scores.shape)
        
test_two_layer_fc_functional()

(64, 10)


In [20]:
input_shape = (32, 32, 3)
hidden_size, num_classes = 4000, 10
learning_rate = 1e-2
print_every=100

def model_init_fn():
    return two_layer_fc_functional(input_shape, hidden_size, num_classes)

def optimizer_init_fn():
    return tf.keras.optimizers.SGD(learning_rate=learning_rate)

train_part34(model_init_fn, optimizer_init_fn)

Iteration 0, Epoch 1, Loss: 3.3149337768554688, Accuracy: 3.125, Val Loss: 2.833019256591797, Val Accuracy: 13.899999618530273
Iteration 100, Epoch 1, Loss: 2.2403507232666016, Accuracy: 28.341585159301758, Val Loss: 1.877618670463562, Val Accuracy: 37.79999923706055
Iteration 200, Epoch 1, Loss: 2.08213210105896, Accuracy: 32.22170639038086, Val Loss: 1.8389972448349, Val Accuracy: 40.099998474121094
Iteration 300, Epoch 1, Loss: 2.0040411949157715, Accuracy: 34.131019592285156, Val Loss: 1.8401671648025513, Val Accuracy: 37.70000076293945
Iteration 400, Epoch 1, Loss: 1.9345605373382568, Accuracy: 36.050498962402344, Val Loss: 1.6983389854431152, Val Accuracy: 42.69999694824219
Iteration 500, Epoch 1, Loss: 1.8884904384613037, Accuracy: 37.04465866088867, Val Loss: 1.6427656412124634, Val Accuracy: 43.70000076293945
Iteration 600, Epoch 1, Loss: 1.859312653541565, Accuracy: 37.88737487792969, Val Loss: 1.6761571168899536, Val Accuracy: 42.39999771118164
Iteration 700, Epoch 1, Loss: 

Поэкспериментируйте с архитектурой сверточной сети. Для вашего набора данных вам необходимо получить как минимум 70% accuracy на валидационной выборке за 10 эпох обучения. Опишите все эксперименты и сделайте выводы (без выполнения данного пункта работы приниматься не будут). 

Эспериментируйте с архитектурой, гиперпараметрами, функцией потерь, регуляризацией, методом оптимизации.  

https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/BatchNormalization#methods https://www.tensorflow.org/versions/r2.0/api_docs/python/tf/keras/layers/Dropout#methods

In [23]:
class CustomConvNet(tf.keras.Model):
    def __init__(self):
        super(CustomConvNet, self).__init__()
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        initializer = tf.initializers.VarianceScaling(scale=2.0)

        self.conv1 = tf.keras.layers.Conv2D(32, (3, 3), padding='same',
                                            kernel_initializer=initializer)
        self.bn1 = tf.keras.layers.BatchNormalization()

        self.conv2 = tf.keras.layers.Conv2D(32, (3, 3), padding='same',
                                            kernel_initializer=initializer)
        self.bn2 = tf.keras.layers.BatchNormalization()
        self.pool1 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.drop1 = tf.keras.layers.Dropout(0.2)

        self.conv3 = tf.keras.layers.Conv2D(64, (3, 3), padding='same',
                                            kernel_initializer=initializer)
        self.bn3 = tf.keras.layers.BatchNormalization()

        self.conv4 = tf.keras.layers.Conv2D(64, (3, 3), padding='same',
                                            kernel_initializer=initializer)
        self.bn4 = tf.keras.layers.BatchNormalization()
        self.pool2 = tf.keras.layers.MaxPooling2D(pool_size=(2, 2))
        self.drop2 = tf.keras.layers.Dropout(0.3)

        self.flatten = tf.keras.layers.Flatten()

        self.fc1 = tf.keras.layers.Dense(256, activation='relu',
                                         kernel_initializer=initializer)
        self.drop3 = tf.keras.layers.Dropout(0.4)

        self.fc2 = tf.keras.layers.Dense(10, activation='softmax',
                                         kernel_initializer=initializer)
        
        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
    
    def call(self, input_tensor, training=False):
        ############################################################################
        # TODO: Construct a model that performs well on CIFAR-10                   #
        ############################################################################
        # *****START OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****

        x = self.conv1(input_tensor)
        x = self.bn1(x, training=training)
        x = tf.nn.relu(x)

        x = self.conv2(x)
        x = self.bn2(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool1(x)
        x = self.drop1(x, training=training)

        x = self.conv3(x)
        x = self.bn3(x, training=training)
        x = tf.nn.relu(x)

        x = self.conv4(x)
        x = self.bn4(x, training=training)
        x = tf.nn.relu(x)
        x = self.pool2(x)
        x = self.drop2(x, training=training)

        x = self.flatten(x)
        x = self.fc1(x)
        x = self.drop3(x, training=training)
        x = self.fc2(x)

        # *****END OF YOUR CODE (DO NOT DELETE/MODIFY THIS LINE)*****
        ############################################################################
        #                            END OF YOUR CODE                              #
        ############################################################################
        return x


print_every = 700
num_epochs = 10

model = CustomConvNet()

def model_init_fn():
    return CustomConvNet()

def optimizer_init_fn():
    learning_rate = 1e-3
    return tf.keras.optimizers.Adam(learning_rate) 

train_part34(model_init_fn, optimizer_init_fn, num_epochs=num_epochs, is_training=True)

Iteration 0, Epoch 1, Loss: 3.9118998050689697, Accuracy: 10.9375, Val Loss: 5.384417533874512, Val Accuracy: 11.200000762939453
Iteration 700, Epoch 1, Loss: 1.8357255458831787, Accuracy: 32.649784088134766, Val Loss: 1.4204434156417847, Val Accuracy: 48.400001525878906
Iteration 1400, Epoch 2, Loss: 1.4560656547546387, Accuracy: 45.74311065673828, Val Loss: 1.2215323448181152, Val Accuracy: 57.20000457763672
Iteration 2100, Epoch 3, Loss: 1.3293546438217163, Accuracy: 50.749671936035156, Val Loss: 1.1451568603515625, Val Accuracy: 59.70000076293945
Iteration 2800, Epoch 4, Loss: 1.228868007659912, Accuracy: 55.187625885009766, Val Loss: 0.9987270832061768, Val Accuracy: 64.0999984741211
Iteration 3500, Epoch 5, Loss: 1.150998830795288, Accuracy: 58.116416931152344, Val Loss: 0.9419307708740234, Val Accuracy: 67.5999984741211
Iteration 4200, Epoch 6, Loss: 1.1032565832138062, Accuracy: 59.95199203491211, Val Loss: 0.8919546008110046, Val Accuracy: 69.70000457763672
Iteration 4900, Epo

Опишите все эксперименты, результаты. Сделайте выводы.

В ходе работы исследовалось влияние архитектуры нейронной сети, способа реализации и параметров обучения на качество классификации изображений CIFAR-10. В 1 эксперименте использовалась простая двухслойная полносвязная сеть с оптимизатором SGD и learning rate 1e-2. После 1 эпохи обучения модель показала низкое качество (около 43% на валидационной выборке), что свидетельствует о недообучении. Это связано с тем, что полносвязная архитектура не учитывает пространственные зависимости в изображениях, а также имеет низкую скорость сходимости SGD без дополнительных механизмов ускорения.

Во 2 эксперименте была использована сверточная сеть (ThreeLayerConvNet) с оптимизатором SGD, ускорениями momentum и Nesterov. Это привело к заметному увеличению качества до 56% на валидации уже после 1 эпохи обучения. Сверточные слои эффективно извлекают локальные признаки изображений, а momentum снижает колебания при обновлении весов и ускоряет сходимость, в то время как Nesterov позволяет более точно корректировать направление градиентного спуска.

В 3 эксперименте была рассмотрена та же полносвязная модель, реализованная через API tf.keras.Sequential. Результаты практически совпали с 1 экспериментом (44%), что подтверждает, что способ реализации модели не влияет на качество при фиксированной архитектуре.

В 4 эксперименте сверточная сеть была реализована через Sequential API и обучена двумя способами: с использованием функции train_part34 и через compile/fit. Полученные значения (49–50%) оказались ниже, чем во 2 эксперименте. Это связано с применением более простой архитектуры без дополнительных механизмов стабилизации обучения, таких как Batch Normalization или регуляризации. При этом способ обучения (ручной цикл или встроенный fit) не оказал существенного влияния на итоговое качество.

В 5 эксперименте была реализована более сложная сверточная сеть (CustomConvNet), включающая дополнительные сверточные слои, Batch Normalization, функции активации ReLU, слои подвыборки (Pooling), Dropout и обучаемая с использованием оптимизатора Adam в течение 10 эпох. В результате удалось достичь 75% точности на валидационной выборке. Рост качества обусловлен более глубокой архитектурой, Batch Normalization стабилизирует распределение активаций и ускоряет обучение, Dropout снижает переобучение, а Adam автоматически подбирает эффективный шаг обучения для каждого параметра, за большее количество эпох модель успевает лучше обучиться. 

Результаты экспериментов показывают, что ключевыми факторами повышения качества являются переход к сверточным архитектурам, использование методов ускорения обучения (momentum, Adam) и применение регуляризации. Полученная итоговая модель удовлетворяет требованиям задания, демонстрируя точность выше 70% на валидационной выборке.
